# Proposal 2: MoE Iterative Restoration

This notebook trains the MoE Router end-to-end after the 4 experts have been pre-trained.

Before running this, make sure you have generated the isolated datasets and pre-trained the experts:

```bash
# 1. Generate data
bash generate_isolated_data.sh

# 2. Pre-train experts
conda run -n ml_env python train_expert.py --expert-type upsample --data-dir data/upsample_only --epochs 40
conda run -n ml_env python train_expert.py --expert-type deblur --data-dir data/blur_only --epochs 40
conda run -n ml_env python train_expert.py --expert-type gaussian --data-dir data/gaussian_only --epochs 40
conda run -n ml_env python train_expert.py --expert-type speckle --data-dir data/speckle_only --epochs 40
```

In [1]:
import os
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.4'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import jax
import jax.numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp
from flax import nnx
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

from dataloader import create_src_dataloader
from train_utils import mixed_loss
from moe_model import IterativeMoE
from train_moe import load_expert_checkpoint, asinh_normalize
from evaluator import ModelEvaluator

ModuleNotFoundError: No module named 'BaselineTesting_V2'

In [ ]:
# Config
BATCH_SIZE = 8
EPOCHS = 20
LR = 5e-4
SEED = 42
TAU_START = 2.0
TAU_END = 0.1
FREEZE_EXPERTS = False

In [ ]:
data_dir = Path("../train")
all_noisy = sorted((data_dir / "NoisyLR").glob("*.npy"))
all_gt = sorted((data_dir / "GT").glob("*.npy"))

train_noisy, val_noisy, train_gt, val_gt = train_test_split(
    all_noisy, all_gt, test_size=0.1, random_state=SEED
)

_, train_loader = create_src_dataloader(train_noisy, train_gt, batch_size=BATCH_SIZE, worker_count=1, seed=SEED, shuffle=True, augment=True)
_, val_loader = create_src_dataloader(val_noisy, val_gt, batch_size=BATCH_SIZE, worker_count=1, seed=SEED+1, shuffle=False, augment=False)

print(f"Train pairs: {len(train_noisy)} | Val pairs: {len(val_noisy)}")

In [ ]:
rngs = nnx.Rngs(SEED)
model = IterativeMoE(rngs=rngs)

# Load experts (requires the expert training scripts to be run first)
try:
    load_expert_checkpoint("ckpt/upsample", model.expert_upsample)
    load_expert_checkpoint("ckpt/deblur", model.expert_deblur)
    load_expert_checkpoint("ckpt/gaussian", model.expert_gaussian)
    load_expert_checkpoint("ckpt/speckle", model.expert_speckle)
    print("✅ Loaded all pre-trained experts")
except Exception as e:
    print(f"⚠️ Error loading experts: {e}\nMake sure to run the expert training scripts first.")

In [ ]:
if FREEZE_EXPERTS:
    optimizer = nnx.Optimizer(model.router, optax.adamw(learning_rate=LR))
else:
    optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=LR))

In [ ]:
@nnx.jit
def train_step(model, optimizer, x_norm, gt, key, tau):
    def loss_fn(model):
        pred, _ = model(x_norm, key, tau=tau)
        return mixed_loss(pred, gt)
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

@nnx.jit
def val_step(model, x_norm, gt, key):
    pred, decisions = model(x_norm, key, tau=0.1, deterministic=True)
    return mixed_loss(pred, gt), decisions

In [ ]:
key = jax.random.key(SEED)
train_steps = len(train_noisy) // BATCH_SIZE
val_steps = len(val_noisy) // BATCH_SIZE

for epoch in range(EPOCHS):
    tau = TAU_START + (TAU_END - TAU_START) * (epoch / max(EPOCHS - 1, 1))
    
    epoch_train_losses = []
    for _ in tqdm(range(train_steps), desc=f"Epoch {epoch}"):
        key, subkey = jax.random.split(key)
        batch = next(train_loader)
        x_norm = asinh_normalize(batch["noisy_lr"].astype(jnp.float32), axis=(1, 2))
        loss = train_step(model, optimizer, x_norm, batch["gt"].astype(jnp.float32), subkey, tau)
        epoch_train_losses.append(float(loss))
        
    epoch_val_losses = []
    all_decisions = []
    for _ in range(val_steps):
        key, subkey = jax.random.split(key)
        batch = next(val_loader)
        x_norm = asinh_normalize(batch["noisy_lr"].astype(jnp.float32), axis=(1, 2))
        vloss, decisions = val_step(model, x_norm, batch["gt"].astype(jnp.float32), subkey)
        epoch_val_losses.append(float(vloss))
        all_decisions.append(np.array(decisions))
        
    all_dec = np.concatenate(all_decisions, axis=0)
    counts = np.bincount(all_dec.flatten(), minlength=4)
    routing = " | ".join(f"{n}:{c}" for n, c in zip(["up", "blur", "gauss", "speckle"], counts))
    
    print(f"Epoch {epoch}: Train {np.mean(epoch_train_losses):.4f} | Val {np.mean(epoch_val_losses):.4f} | tau {tau:.2f}")
    print(f"Routing: {routing}")